# Session 4: Deploying Models On-Premise

**Module 5: Deep Learning in Production**  
**SMU Advanced Certificate in Generative AI and Deep Learning**

---

**Duration:** 4 hours (60 min theory, 180 min hands-on)  
**Prerequisites:** Basic Python knowledge, Sessions 1–3 of this module  

**What you will learn:**
- Why and when to deploy models on your own hardware instead of the cloud
- How to run LLMs locally using Ollama
- How to use Apple’s MLX framework for inference on Apple Silicon
- How to build a simple local inference server with FastAPI

**Session outline:**

| Section | Duration | Content |
|---------|----------|---------|
| Theory | ~60 min | On-premise deployment concepts, hardware options, frameworks |
| Part A | ~60 min | Running LLMs with Ollama |
| Part B | ~60 min | Apple MLX Framework |
| Part C | ~60 min | Serving models locally with FastAPI |

---

## Setup: Install Required Packages

Run the cell below to install all packages needed for this session.

📝 **Note:** Some packages are platform-specific:
- `mlx` and `mlx-lm` **only work on Apple Silicon Macs** (M1/M2/M3/M4)
- All other packages work on Mac, Linux, and Windows
- Ollama must be installed separately (instructions in Part A)

In [1]:
# Install required packages
# Note: mlx and mlx-lm will only install on Apple Silicon Macs.
# If you are on Linux/Windows, these lines will fail — that is OK.

!pip install requests fastapi uvicorn psutil aiohttp -q

# Apple Silicon only — safe to skip on other platforms
try:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlx", "mlx-lm", "-q"])
    print("MLX installed successfully (Apple Silicon detected).")
except Exception:
    print("MLX installation skipped — not on Apple Silicon or not supported.")

MLX installed successfully (Apple Silicon detected).


---

# Part 1: Theory — On-Premise Deployment (~60 minutes)

---

## 1.1 Why Deploy On-Premise?

So far in this module, we have looked at deploying models to the **cloud** — services like AWS SageMaker, Google Cloud, or managed API providers. But there are many situations where you need to run models **on your own hardware**.

### When does on-premise make sense?

**Data Privacy and Regulatory Requirements**

Some industries have strict rules about where data can go:

- **Healthcare:** Patient records cannot leave hospital networks in many countries (e.g., HIPAA in the US, PDPA in Singapore)
- **Finance:** Trading algorithms and proprietary data must stay within the firm
- **Government:** Classified or sensitive data cannot be sent to third-party cloud providers
- **Legal:** Client-attorney privilege may be compromised if data flows through external APIs

If your data **cannot leave your network**, you must bring the model to the data — not the data to the model.

**Latency-Sensitive Applications**

Every cloud API call involves a network round-trip:

```
Your App  →  Internet  →  Cloud Provider  →  Model  →  Internet  →  Your App
              ~10-50ms       ~5-20ms         ~100ms+       ~10-50ms
```

For applications like:
- Real-time video analysis
- Robotics control systems
- High-frequency trading
- Interactive gaming AI

...even 50ms of extra network latency is unacceptable. Running the model locally removes the network round-trip entirely.

**Cost at Scale**

Cloud API pricing works well for low-to-medium volume, but costs can explode:

| Usage Level | Cloud API Cost (est.) | On-Premise Cost (est.) |
|------------|----------------------|----------------------|
| 1,000 requests/day | ~$30/month | Not worth it |
| 100,000 requests/day | ~$3,000/month | ~$500/month (amortised) |
| 1,000,000 requests/day | ~$30,000/month | ~$2,000/month (amortised) |

At high volumes, owning the hardware pays for itself quickly.

### Comparison: Cloud vs On-Premise Deployment

| Factor | Cloud Deployment | On-Premise Deployment |
|--------|-----------------|----------------------|
| **Setup effort** | Low — managed services | High — you manage everything |
| **Scaling** | Easy — auto-scale | Manual — buy more hardware |
| **Data privacy** | Data leaves your network | Data stays on your network |
| **Latency** | Higher (network round-trip) | Lower (local inference) |
| **Cost (low volume)** | Cheaper (pay-per-use) | Expensive (fixed hardware cost) |
| **Cost (high volume)** | Expensive | Cheaper (amortised) |
| **Maintenance** | Provider handles it | You handle updates, monitoring |
| **GPU availability** | Subject to cloud capacity | Always available (you own it) |

💡 **Key Concept:** There is no single “best” option. The right choice depends on your **data sensitivity**, **latency needs**, **volume**, and **budget**. Many organisations use a **hybrid** approach — cloud for bursty workloads, on-premise for steady high-volume or sensitive workloads.

---

## 1.2 Linux/Nvidia Server Deployment

The **standard industry setup** for running ML models in production is:

```
Linux Server + Nvidia GPU(s)
```

This has been the dominant platform for a decade and has the best ecosystem support.

### Key Components

**1. Nvidia GPUs**
- The workhorses of ML inference (and training)
- Common choices: A100, H100 (data centre), RTX 4090 (prosumer)
- GPUs have their own dedicated memory (VRAM) — typically 24GB to 80GB

**2. CUDA and cuDNN**
- **CUDA**: Nvidia’s parallel computing platform — lets code run on the GPU
- **cuDNN**: Nvidia’s deep learning library — optimised operations for neural networks
- Almost all ML frameworks (PyTorch, TensorFlow) use CUDA under the hood

**3. Nvidia Container Runtime (Docker)**
- Run GPU-accelerated containers
- Makes deployment reproducible and portable
- You package your model + serving code in a Docker container

### Common Serving Frameworks

| Framework | Best For | Key Features |
|-----------|----------|-------------|
| **Nvidia Triton Inference Server** | Production multi-model serving | Supports PyTorch, TensorFlow, ONNX; dynamic batching; model versioning |
| **TorchServe** | PyTorch models | PyTorch-native; easy setup; good for PyTorch-only shops |
| **vLLM** | LLM serving | PagedAttention for efficient memory; continuous batching; very fast |
| **Text Generation Inference (TGI)** | LLM serving | By Hugging Face; production-ready; good Hugging Face integration |

### Typical GPU Server Architecture

```
┌─────────────────────────────────────────────────┐
│                  GPU SERVER                       │
│                                                   │
│  ┌──────────────┐    ┌──────────────────────┐    │
│  │   CPU + RAM   │    │   Nvidia GPU(s)       │    │
│  │  (64-512 GB)  │◄──►│  (24-80 GB VRAM each) │    │
│  └──────┬───────┘    └──────────────────────┘    │
│         │                                         │
│  ┌──────▼───────────────────────────────────┐    │
│  │          Docker + Nvidia Runtime           │    │
│  │  ┌─────────────────────────────────────┐  │    │
│  │  │   Triton / vLLM / TorchServe        │  │    │
│  │  │   (Serving Framework)                │  │    │
│  │  │                                      │  │    │
│  │  │   Model A    Model B    Model C      │  │    │
│  │  └─────────────────────────────────────┘  │    │
│  └─────────────────────┬───────────────────┘    │
│                      │                             │
│               REST / gRPC API                      │
└──────────────────────┼─────────────────────────┘
                       │
              ┌────────▼────────┐
              │  Your Application │
              └─────────────────┘
```

📝 **Note:** You do not need a GPU server to follow this session. We will use **Ollama** and **MLX**, which run on consumer hardware (including laptops). The concepts transfer directly to production GPU servers.

---

## 1.3 Apple Silicon for ML — The Alternative

Since 2020, Apple has been shipping its own chips (M1, M2, M3, M4 series) that have a fundamentally different architecture from traditional CPU+GPU setups.

### Unified Memory Architecture

In a **traditional** system:
```
CPU  ◄── System RAM (e.g., 64 GB) ──►  copy data back and forth  ◄── GPU VRAM (e.g., 24 GB)
```

On **Apple Silicon**:
```
CPU + GPU + Neural Engine  ◄──── Shared Unified Memory (e.g., 24–192 GB) ────►
```

**Why this matters for ML:**
- No need to copy data between CPU and GPU memory — it is **shared**
- A MacBook Pro with 96GB unified memory can load a model that would need a very expensive GPU
- An Nvidia RTX 4090 has 24GB VRAM. An M4 Max MacBook can have 128GB of unified memory
- For **inference** (not training), Apple Silicon is surprisingly competitive

### MLX Framework

**MLX** is Apple’s open-source ML framework, designed specifically for Apple Silicon:

- **NumPy-like API** — if you know NumPy, you can use MLX
- **Lazy evaluation** — operations are only computed when the result is needed (efficient)
- **Unified memory** — arrays live in shared memory accessible by both CPU and GPU
- **Composable transformations** — like JAX, supports `grad()`, `vmap()`, etc.

```python
import mlx.core as mx

# Looks just like NumPy
a = mx.array([1, 2, 3])
b = mx.array([4, 5, 6])
c = a + b  # Runs on GPU automatically on Apple Silicon
```

### Ollama

**Ollama** is a tool that makes running LLMs locally as easy as running a Docker container:

- One-command install, one-command model download
- Works on **Mac** (Apple Silicon), **Linux**, and **Windows**
- Automatically uses GPU acceleration (Metal on Mac, CUDA on Nvidia)
- Provides a local REST API compatible with the OpenAI API format
- Great for development and testing

### When to Use What?

| Scenario | Best Choice |
|----------|------------|
| Production, high-throughput serving | Nvidia GPU + Triton/vLLM |
| Development & testing on Mac | Ollama or MLX |
| Running large models (70B+) on Mac | MLX (can use all unified memory) |
| Privacy-sensitive local inference | Ollama (any platform) |
| Training models | Nvidia GPU (still the standard) |
| Budget-friendly inference | Apple Silicon Mac (good performance per dollar) |

💡 **Key Concept:** Apple Silicon is not a replacement for Nvidia GPUs in production data centres. But for **local development**, **edge deployment**, and **privacy-sensitive applications**, it is an excellent and cost-effective option.

---

## 1.4 Distributed Inference

Some models are simply **too large** to fit on a single device. For example:

| Model | Parameters | Memory Needed (FP16) |
|-------|-----------|---------------------|
| Llama 3.2 1B | 1 billion | ~2 GB |
| Llama 3.1 8B | 8 billion | ~16 GB |
| Llama 3.1 70B | 70 billion | ~140 GB |
| Llama 3.1 405B | 405 billion | ~810 GB |

A single A100 GPU has 80GB of VRAM. To run a 405B model, you would need **at least 11 A100s**.

### Model Parallelism vs Data Parallelism

**Data Parallelism** — same model on multiple devices, different data:
```
Device 1: Full Model  ← Batch of requests A
Device 2: Full Model  ← Batch of requests B
Device 3: Full Model  ← Batch of requests C
```
Used to increase **throughput** (handle more requests per second).

**Model Parallelism (Tensor Parallelism)** — model is split across devices:
```
Device 1: Layers 1-10   ← Input flows through
Device 2: Layers 11-20  ← then here
Device 3: Layers 21-30  ← then here → Output
```
Used when a model is **too large** for one device.

### Tools for Distributed Inference

- **vLLM**: supports tensor parallelism across multiple GPUs
- **Text Generation Inference (TGI)**: Hugging Face’s solution, supports sharding
- **DeepSpeed Inference**: Microsoft’s library for distributed inference
- **MLX distributed** (experimental): for splitting models across Apple Silicon devices

📝 **Note:** Distributed inference is an advanced topic. For this session, we focus on **single-device** inference, which covers the majority of use cases for models up to ~70B parameters (with quantisation).

---

## 1.5 Choosing Your Deployment Target

Use this decision framework when planning an on-premise deployment:

```
                        ┌─────────────────────┐
                        │  What is your model  │
                        │      size?           │
                        └─────────┬───────────┘
                                  │
                    ┌─────────────┼─────────────┐
                    │             │              │
                 Small         Medium          Large
               (< 3B)        (3B–13B)        (13B+)
                    │             │              │
                    ▼             ▼              ▼
              Any device     GPU recommended   GPU required
              CPU is fine    or Apple Silicon   Nvidia or Apple
                                               Silicon w/ lots
                                               of RAM
```

### Summary of Options

| Option | Hardware | Cost | Setup Effort | Best For |
|--------|----------|------|-------------|----------|
| **Ollama** | Any (CPU/GPU) | Free | Very Low | Development, testing, small deployments |
| **MLX** | Apple Silicon Mac | Mac cost | Low | Local inference, Apple ecosystem |
| **vLLM** | Nvidia GPU | GPU cost | Medium | High-throughput LLM serving |
| **Triton** | Nvidia GPU | GPU cost | High | Multi-model production serving |
| **TorchServe** | Any (GPU preferred) | Varies | Medium | PyTorch model serving |

💡 **Key Concept:** Start simple. **Ollama** is perfect for getting started and prototyping. Move to **vLLM** or **Triton** when you need production-grade performance. Use **MLX** when you want to leverage Apple Silicon hardware.

---

# Part 2: Hands-On Exercises (~180 minutes)

Now let us put theory into practice. We will:
1. Run LLMs locally with Ollama
2. Use Apple’s MLX framework (Apple Silicon only)
3. Build a local inference server with FastAPI

⚠️ **Important:**
- **Ollama exercises** work on Mac, Linux, and Windows
- **MLX exercises** require an **Apple Silicon Mac** (M1/M2/M3/M4)
- If you are not on Apple Silicon, read through the MLX section and focus on the Ollama and FastAPI exercises

---

## Part A: Running LLMs with Ollama (~60 min)

### What is Ollama?

Ollama is a tool that packages LLMs with their runtime into a single, easy-to-use application. Think of it like Docker, but for language models.

---

### 🔧 Exercise 1: Installing and Running Ollama

**Step 1: Install Ollama**

Ollama must be installed as a system application (not a Python package).

**On macOS:**
- Download from [https://ollama.com](https://ollama.com) and install the app, **or**
- Use Homebrew: `brew install ollama`

**On Linux:**
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

**On Windows:**
- Download from [https://ollama.com](https://ollama.com) and run the installer

After installation, Ollama runs as a background service and exposes an API on `http://localhost:11434`.

⚠️ **Important:** Make sure Ollama is running before proceeding. On macOS, launch the Ollama app. On Linux, it starts automatically as a service.

In [2]:
# Check if Ollama is running
import subprocess
import requests

def check_ollama():
    """Check if Ollama is installed and running."""
    # Check if the ollama command exists
    try:
        result = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
        print(f"Ollama version: {result.stdout.strip()}")
    except FileNotFoundError:
        print("ERROR: Ollama is not installed. Please install it from https://ollama.com")
        return False

    # Check if the Ollama server is running
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get("models", [])
            print(f"Ollama server is running. {len(models)} model(s) available.")
            return True
    except requests.ConnectionError:
        print("Ollama is installed but the server is not running.")
        print("Start it by launching the Ollama app (macOS) or running 'ollama serve' (Linux).")
        return False

check_ollama()

ERROR: Ollama is not installed. Please install it from https://ollama.com


False

**Step 2: Pull a Small Model**

We will start with a small model that runs well even on laptops without a dedicated GPU. **Llama 3.2 1B** is only ~1.3 GB and runs on almost any modern machine.

Run the cell below to download it. This may take a few minutes depending on your internet speed.

In [3]:
# Pull a small model — this downloads the model weights
# Llama 3.2 1B is ~1.3 GB, small enough for most machines

import subprocess
import sys

print("Downloading llama3.2:1b — this may take a few minutes...")
print("=" * 60)

process = subprocess.Popen(
    ["ollama", "pull", "llama3.2:1b"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode == 0:
    print("\nModel downloaded successfully!")
else:
    print("\nError downloading model. Make sure Ollama is running.")

FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

**Step 3: Chat with the Model from Python**

We can interact with Ollama’s API using Python’s `requests` library. Ollama provides an API that is compatible with the OpenAI chat format.

In [4]:
# Chat with the model using Ollama's API
import requests
import json
import time

def chat_with_ollama(prompt, model="llama3.2:1b"):
    """Send a prompt to Ollama and get a response."""
    url = "http://localhost:11434/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False  # Get the complete response at once
    }

    start_time = time.time()
    response = requests.post(url, json=payload, timeout=120)
    elapsed = time.time() - start_time

    if response.status_code == 200:
        result = response.json()
        return {
            "response": result["response"],
            "total_duration_ms": result.get("total_duration", 0) / 1e6,
            "eval_count": result.get("eval_count", 0),
            "wall_time_s": elapsed
        }
    else:
        return {"error": f"Status {response.status_code}: {response.text}"}

# Try a simple conversation
result = chat_with_ollama("Explain what machine learning is in 2-3 sentences, suitable for a beginner.")

print("Response:")
print("-" * 60)
print(result["response"])
print("-" * 60)
print(f"\nTokens generated: {result['eval_count']}")
print(f"Wall clock time: {result['wall_time_s']:.2f} seconds")
if result['eval_count'] > 0:
    tokens_per_sec = result['eval_count'] / result['wall_time_s']
    print(f"Speed: {tokens_per_sec:.1f} tokens/second")

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x108465610>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [5]:
# Try a few more prompts to get a feel for the model
prompts = [
    "What is the capital of Singapore?",
    "Write a Python function that checks if a number is prime.",
    "What are three benefits of deploying ML models on-premise?"
]

for i, prompt in enumerate(prompts, 1):
    print(f"\n{'='*60}")
    print(f"Prompt {i}: {prompt}")
    print(f"{'='*60}")
    result = chat_with_ollama(prompt)
    print(result["response"][:500])  # Truncate long responses
    print(f"\n[{result['eval_count']} tokens in {result['wall_time_s']:.2f}s]")


Prompt 1: What is the capital of Singapore?


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10e66a3c0>: Failed to establish a new connection: [Errno 61] Connection refused'))

---

### 🔧 Exercise 2: Ollama as a Local API Server

Ollama automatically runs a REST API server on `http://localhost:11434`. This means any application on your machine can send requests to it — just like calling a cloud API, but everything stays local.

This is the foundation of on-premise deployment: instead of calling `api.openai.com`, your application calls `localhost:11434`.

In [6]:
# Exercise 2a: Explore the Ollama API endpoints
import requests
import json

BASE_URL = "http://localhost:11434"

# List available models
print("Available models:")
print("=" * 60)
response = requests.get(f"{BASE_URL}/api/tags")
models = response.json().get("models", [])

for model in models:
    name = model["name"]
    size_gb = model.get("size", 0) / (1024**3)
    print(f"  - {name} ({size_gb:.2f} GB)")

print(f"\nTotal models: {len(models)}")

Available models:


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10e7b7f80>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [7]:
# Exercise 2b: Use the OpenAI-compatible chat endpoint
# Ollama supports the OpenAI chat completions format — this means
# code written for OpenAI's API can switch to Ollama with minimal changes.

import requests
import json

def chat_openai_format(messages, model="llama3.2:1b"):
    """Use Ollama's OpenAI-compatible endpoint."""
    url = "http://localhost:11434/v1/chat/completions"

    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.7
    }

    response = requests.post(url, json=payload, timeout=120)
    return response.json()

# Same format as the OpenAI API
messages = [
    {"role": "system", "content": "You are a helpful assistant that explains things simply."},
    {"role": "user", "content": "What is an API? Explain in one paragraph."}
]

result = chat_openai_format(messages)

print("Response (OpenAI-compatible format):")
print("-" * 60)
print(result["choices"][0]["message"]["content"])
print("-" * 60)
print(f"\nModel: {result['model']}")
print(f"Tokens used: {result['usage']}")

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /v1/chat/completions (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10e7b74a0>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [8]:
# Exercise 2c: Streaming responses
# Streaming sends tokens as they are generated, giving a "typing" effect.
# This is important for user-facing applications — users don't want to wait
# for the entire response before seeing anything.

import requests
import json
import time
import sys

def stream_response(prompt, model="llama3.2:1b"):
    """Stream a response from Ollama, printing tokens as they arrive."""
    url = "http://localhost:11434/api/generate"

    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True  # Enable streaming
    }

    print("Streaming response:")
    print("-" * 60)

    start_time = time.time()
    token_count = 0

    response = requests.post(url, json=payload, stream=True, timeout=120)

    for line in response.iter_lines():
        if line:
            data = json.loads(line)
            token = data.get("response", "")
            print(token, end="", flush=True)  # Print each token immediately
            token_count += 1

            if data.get("done", False):
                break

    elapsed = time.time() - start_time
    print(f"\n{'=' * 60}")
    print(f"Streamed {token_count} tokens in {elapsed:.2f}s ({token_count/elapsed:.1f} tok/s)")

stream_response("Give me 5 tips for deploying machine learning models in production. Be concise.")

Streaming response:
------------------------------------------------------------


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10e869220>: Failed to establish a new connection: [Errno 61] Connection refused'))

---

## Part B: Apple MLX Framework (~60 min)

⚠️ **Important:** The exercises in this section **require an Apple Silicon Mac** (M1, M2, M3, or M4 chip). If you are on a different platform, read through the code and explanations to understand the concepts — the ideas apply broadly, even if the specific framework does not run on your machine.

---

### 🔧 Exercise 3: Getting Started with MLX

In [9]:
# Exercise 3a: Check if MLX is available
import platform
import sys

print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")
print(f"Python: {sys.version}")

try:
    import mlx.core as mx
    print(f"\nMLX version: {mx.__version__}")
    print(f"Default device: {mx.default_device()}")
    print("MLX is ready to use!")
    MLX_AVAILABLE = True
except ImportError:
    print("\nMLX is not available on this system.")
    print("MLX requires an Apple Silicon Mac (M1/M2/M3/M4).")
    print("You can still read through the exercises to learn the concepts.")
    MLX_AVAILABLE = False

Platform: macOS-26.3.1-arm64-arm-64bit
Processor: arm
Python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]



MLX version: 0.31.1
Default device: Device(gpu, 0)
MLX is ready to use!


In [10]:
# Exercise 3b: MLX basics — it looks just like NumPy
# This cell demonstrates that MLX has a familiar API

if MLX_AVAILABLE:
    import mlx.core as mx
    import time

    # Create arrays (just like NumPy)
    a = mx.array([1.0, 2.0, 3.0, 4.0, 5.0])
    b = mx.array([10.0, 20.0, 30.0, 40.0, 50.0])

    # Basic operations
    c = a + b
    d = mx.sqrt(a)
    e = mx.matmul(
        mx.random.normal((100, 100)),
        mx.random.normal((100, 100))
    )

    # Force evaluation (MLX uses lazy evaluation)
    mx.eval(c, d, e)

    print("MLX Array examples:")
    print(f"  a = {a}")
    print(f"  b = {b}")
    print(f"  a + b = {c}")
    print(f"  sqrt(a) = {d}")
    print(f"  Matrix multiply shape: {e.shape}")

    # Benchmark: matrix multiplication
    print("\nBenchmark: 1000x1000 matrix multiplication")
    x = mx.random.normal((1000, 1000))
    y = mx.random.normal((1000, 1000))

    # Warm up
    mx.eval(mx.matmul(x, y))

    start = time.time()
    for _ in range(10):
        result = mx.matmul(x, y)
        mx.eval(result)
    elapsed = time.time() - start

    print(f"  10 multiplications in {elapsed:.3f}s ({elapsed/10*1000:.1f}ms each)")
else:
    print("Skipping — MLX not available. Here is what this code does:")
    print("  - Creates MLX arrays (similar to NumPy)")
    print("  - Performs basic operations: addition, sqrt, matrix multiplication")
    print("  - Benchmarks 1000x1000 matrix multiplication")
    print("  - On Apple Silicon, these operations run on the GPU automatically")

MLX Array examples:
  a = array([1, 2, 3, 4, 5], dtype=float32)
  b = array([10, 20, 30, 40, 50], dtype=float32)
  a + b = array([11, 22, 33, 44, 55], dtype=float32)
  sqrt(a) = array([1, 1.41421, 1.73205, 2, 2.23607], dtype=float32)
  Matrix multiply shape: (100, 100)

Benchmark: 1000x1000 matrix multiplication
  10 multiplications in 0.025s (2.5ms each)


**Loading a Language Model with MLX**

The `mlx-lm` package makes it easy to load and run language models from Hugging Face in MLX format. Many popular models have been converted to MLX format by the community.

We will load a small quantised model — **Qwen2.5-0.5B** — which is only ~400MB and runs fast even on entry-level Apple Silicon.

In [11]:
# Exercise 3c: Load and run a language model with MLX
# This downloads a small quantised model and generates text

if MLX_AVAILABLE:
    from mlx_lm import load, generate
    import time

    print("Loading Qwen2.5-0.5B-Instruct (4-bit quantised)...")
    print("This will download the model on first run (~400 MB).")
    print()

    # Load a small quantised model
    model, tokenizer = load("mlx-community/Qwen2.5-0.5B-Instruct-4bit")

    print("Model loaded successfully!")
    print()

    # Generate text
    prompt = "Explain what deployment means in machine learning in 2 sentences."

    # Format as a chat message
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print(f"Prompt: {prompt}")
    print("-" * 60)

    start_time = time.time()
    response = generate(
        model,
        tokenizer,
        prompt=formatted_prompt,
        max_tokens=200,
        verbose=False
    )
    elapsed = time.time() - start_time

    print(f"Response: {response}")
    print("-" * 60)

    # Estimate tokens per second
    num_tokens = len(tokenizer.encode(response))
    print(f"\nGenerated ~{num_tokens} tokens in {elapsed:.2f}s")
    print(f"Speed: ~{num_tokens/elapsed:.1f} tokens/second")
else:
    print("Skipping — MLX not available.")
    print("This exercise loads Qwen2.5-0.5B (a small language model) using MLX")
    print("and generates a response, measuring tokens per second.")

ModuleNotFoundError: No module named 'mlx_lm'

In [12]:
# Exercise 3d: Measure generation speed with different prompt lengths
# This helps understand how model performance varies

if MLX_AVAILABLE:
    from mlx_lm import load, generate
    import time

    # Use the model already loaded above (or reload if needed)
    try:
        _ = model
    except NameError:
        model, tokenizer = load("mlx-community/Qwen2.5-0.5B-Instruct-4bit")

    prompts = [
        "What is AI?",
        "Explain the difference between machine learning and deep learning in detail.",
        "Write a step-by-step guide for deploying a machine learning model on a local server. Include hardware requirements, software setup, and testing procedures."
    ]

    print("Measuring generation speed across different prompts:")
    print("=" * 60)

    for i, prompt in enumerate(prompts, 1):
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        start = time.time()
        response = generate(model, tokenizer, prompt=formatted, max_tokens=100, verbose=False)
        elapsed = time.time() - start

        num_tokens = len(tokenizer.encode(response))
        print(f"\nPrompt {i}: \"{prompt[:50]}...\"")
        print(f"  Output tokens: ~{num_tokens}")
        print(f"  Time: {elapsed:.2f}s")
        print(f"  Speed: ~{num_tokens/elapsed:.1f} tokens/second")
else:
    print("Skipping — MLX not available.")

ModuleNotFoundError: No module named 'mlx_lm'

---

### 🔧 Exercise 4: MLX vs PyTorch Comparison

Let us compare MLX and PyTorch for a simple task to understand their performance characteristics on Apple Silicon.

In [13]:
# Exercise 4: Compare MLX and PyTorch on simple operations
import time
import psutil

def get_memory_mb():
    """Get current process memory usage in MB."""
    process = psutil.Process()
    return process.memory_info().rss / (1024 * 1024)

print("=" * 60)
print("MLX vs PyTorch Comparison on Apple Silicon")
print("=" * 60)

# --- PyTorch benchmark ---
print("\n--- PyTorch (CPU) ---")
try:
    import torch

    mem_before = get_memory_mb()

    # Matrix multiplication benchmark
    size = 2000
    a_pt = torch.randn(size, size)
    b_pt = torch.randn(size, size)

    # Warm up
    _ = torch.matmul(a_pt, b_pt)

    start = time.time()
    for _ in range(5):
        c_pt = torch.matmul(a_pt, b_pt)
    pt_time = time.time() - start

    mem_after = get_memory_mb()

    print(f"  {size}x{size} matmul (5 reps): {pt_time:.3f}s ({pt_time/5*1000:.1f}ms each)")
    print(f"  Memory delta: {mem_after - mem_before:.1f} MB")

except ImportError:
    print("  PyTorch not installed — skipping")
    pt_time = None

# --- MLX benchmark ---
if MLX_AVAILABLE:
    import mlx.core as mx

    print("\n--- MLX (GPU on Apple Silicon) ---")

    mem_before = get_memory_mb()

    size = 2000
    a_mx = mx.random.normal((size, size))
    b_mx = mx.random.normal((size, size))

    # Warm up
    mx.eval(mx.matmul(a_mx, b_mx))

    start = time.time()
    for _ in range(5):
        c_mx = mx.matmul(a_mx, b_mx)
        mx.eval(c_mx)
    mx_time = time.time() - start

    mem_after = get_memory_mb()

    print(f"  {size}x{size} matmul (5 reps): {mx_time:.3f}s ({mx_time/5*1000:.1f}ms each)")
    print(f"  Memory delta: {mem_after - mem_before:.1f} MB")

    # Comparison
    if pt_time is not None:
        print(f"\n--- Comparison ---")
        speedup = pt_time / mx_time
        if speedup > 1:
            print(f"  MLX is {speedup:.1f}x faster than PyTorch CPU")
        else:
            print(f"  PyTorch CPU is {1/speedup:.1f}x faster than MLX")
        print("  Note: PyTorch is running on CPU only. With MPS backend, results may differ.")
else:
    print("\n--- MLX ---")
    print("  MLX not available — skipping")

print()
print("Key takeaway: On Apple Silicon, MLX uses the GPU automatically for")
print("array operations, while PyTorch defaults to CPU. MLX's unified memory")
print("means no data copying overhead between CPU and GPU.")

MLX vs PyTorch Comparison on Apple Silicon

--- PyTorch (CPU) ---


  2000x2000 matmul (5 reps): 0.066s (13.2ms each)
  Memory delta: 91.2 MB

--- MLX (GPU on Apple Silicon) ---


  2000x2000 matmul (5 reps): 0.026s (5.2ms each)
  Memory delta: -8.4 MB

--- Comparison ---
  MLX is 2.5x faster than PyTorch CPU
  Note: PyTorch is running on CPU only. With MPS backend, results may differ.

Key takeaway: On Apple Silicon, MLX uses the GPU automatically for
array operations, while PyTorch defaults to CPU. MLX's unified memory
means no data copying overhead between CPU and GPU.


---

## Part C: Serving Models Locally (~60 min)

Now let us build something practical — a **local inference server** using FastAPI that wraps Ollama’s API. This is the foundation of any on-premise deployment.

---

### 🔧 Exercise 5: Building a Local Inference Server

In this exercise, we will create a FastAPI server that:
1. Wraps Ollama’s API with our own endpoint
2. Adds request logging
3. Adds simple rate limiting
4. Formats responses consistently

This is exactly what a real on-premise deployment looks like — you put a custom API layer in front of your model server to add business logic, authentication, logging, and monitoring.

📝 **Note:** We will write the server code to a file and run it in the background, then test it from this notebook.

In [14]:
# Exercise 5a: Write the FastAPI server code to a file

server_code = '''
"""
Local Inference Server
Wraps Ollama's API with logging, rate limiting, and consistent response formatting.
This is the foundation of on-premise model deployment.
"""

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from datetime import datetime, timedelta
from collections import defaultdict
import requests
import json
import time
import logging

# --- Configuration ---
OLLAMA_BASE_URL = "http://localhost:11434"
DEFAULT_MODEL = "llama3.2:1b"
RATE_LIMIT_REQUESTS = 10      # Max requests per window
RATE_LIMIT_WINDOW_SECONDS = 60  # Window size in seconds

# --- Setup logging ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# --- FastAPI app ---
app = FastAPI(
    title="Local Inference Server",
    description="A simple on-premise model serving API wrapping Ollama",
    version="1.0.0"
)

# --- Rate limiter (simple in-memory implementation) ---
request_counts = defaultdict(list)

def check_rate_limit(client_ip: str) -> bool:
    """Check if a client has exceeded the rate limit."""
    now = datetime.now()
    window_start = now - timedelta(seconds=RATE_LIMIT_WINDOW_SECONDS)

    # Remove old entries
    request_counts[client_ip] = [
        t for t in request_counts[client_ip] if t > window_start
    ]

    # Check limit
    if len(request_counts[client_ip]) >= RATE_LIMIT_REQUESTS:
        return False

    request_counts[client_ip].append(now)
    return True

# --- Request/Response models ---
class ChatRequest(BaseModel):
    prompt: str
    model: str = DEFAULT_MODEL
    max_tokens: int = 500
    temperature: float = 0.7

class ChatResponse(BaseModel):
    response: str
    model: str
    tokens_generated: int
    duration_seconds: float
    timestamp: str

# --- Endpoints ---
@app.get("/health")
async def health_check():
    """Check if the server and Ollama are healthy."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        ollama_ok = r.status_code == 200
    except Exception:
        ollama_ok = False

    return {
        "status": "healthy" if ollama_ok else "degraded",
        "ollama_connected": ollama_ok,
        "timestamp": datetime.now().isoformat()
    }

@app.get("/models")
async def list_models():
    """List available models."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        models = r.json().get("models", [])
        return {
            "models": [
                {
                    "name": m["name"],
                    "size_gb": round(m.get("size", 0) / (1024**3), 2)
                }
                for m in models
            ]
        }
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Cannot reach Ollama: {str(e)}")

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest, req: Request):
    """Send a prompt to the model and get a response."""
    client_ip = req.client.host

    # Rate limiting
    if not check_rate_limit(client_ip):
        raise HTTPException(
            status_code=429,
            detail=f"Rate limit exceeded. Max {RATE_LIMIT_REQUESTS} requests per {RATE_LIMIT_WINDOW_SECONDS} seconds."
        )

    # Log the request
    logger.info(f"Request from {client_ip}: model={request.model}, prompt_length={len(request.prompt)}")

    # Call Ollama
    start_time = time.time()
    try:
        r = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json={
                "model": request.model,
                "prompt": request.prompt,
                "stream": False,
                "options": {
                    "num_predict": request.max_tokens,
                    "temperature": request.temperature
                }
            },
            timeout=120
        )
        r.raise_for_status()
    except requests.ConnectionError:
        raise HTTPException(status_code=503, detail="Cannot connect to Ollama. Is it running?")
    except requests.Timeout:
        raise HTTPException(status_code=504, detail="Ollama request timed out")

    duration = time.time() - start_time
    result = r.json()

    # Log the response
    tokens = result.get("eval_count", 0)
    logger.info(f"Response: {tokens} tokens in {duration:.2f}s")

    return ChatResponse(
        response=result["response"],
        model=request.model,
        tokens_generated=tokens,
        duration_seconds=round(duration, 3),
        timestamp=datetime.now().isoformat()
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Write the server code to a file
with open("local_inference_server.py", "w") as f:
    f.write(server_code)

print("Server code written to: local_inference_server.py")
print()
print("The server provides these endpoints:")
print("  GET  /health  - Check server and Ollama status")
print("  GET  /models  - List available models")
print("  POST /chat    - Send a prompt, get a response")
print("  GET  /docs    - Auto-generated API documentation (Swagger)")

Server code written to: local_inference_server.py

The server provides these endpoints:
  GET  /health  - Check server and Ollama status
  GET  /models  - List available models
  POST /chat    - Send a prompt, get a response
  GET  /docs    - Auto-generated API documentation (Swagger)


In [15]:
# Exercise 5b: Start the server in the background
import subprocess
import time
import requests

# Start the server as a background process
print("Starting the local inference server on port 8000...")
server_process = subprocess.Popen(
    ["python", "local_inference_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for the server to start
for i in range(10):
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"Server is running! (took {i+1} seconds to start)")
            print(f"Health check: {r.json()}")
            break
    except requests.ConnectionError:
        time.sleep(1)
else:
    print("Server failed to start. Check if port 8000 is available.")
    stderr = server_process.stderr.read().decode()
    if stderr:
        print(f"Error: {stderr[:500]}")

Starting the local inference server on port 8000...


Server is running! (took 3 seconds to start)
Health check: {'status': 'degraded', 'ollama_connected': False, 'timestamp': '2026-04-08T11:15:34.323494'}


In [16]:
# Exercise 5c: Test the server endpoints
import requests
import json

BASE = "http://localhost:8000"

# Test 1: Health check
print("1. Health Check")
print("-" * 40)
r = requests.get(f"{BASE}/health")
print(json.dumps(r.json(), indent=2))

# Test 2: List models
print("\n2. Available Models")
print("-" * 40)
r = requests.get(f"{BASE}/models")
print(json.dumps(r.json(), indent=2))

# Test 3: Chat
print("\n3. Chat Request")
print("-" * 40)
r = requests.post(f"{BASE}/chat", json={
    "prompt": "What is on-premise deployment? Answer in one sentence.",
    "max_tokens": 100
})
result = r.json()
print(f"Response: {result['response']}")
print(f"Tokens: {result['tokens_generated']}")
print(f"Duration: {result['duration_seconds']}s")

1. Health Check
----------------------------------------
{
  "status": "degraded",
  "ollama_connected": false,
  "timestamp": "2026-04-08T11:15:34.335296"
}

2. Available Models
----------------------------------------
{
  "detail": "Cannot reach Ollama: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x109958740>: Failed to establish a new connection: [Errno 61] Connection refused'))"
}

3. Chat Request
----------------------------------------


KeyError: 'response'

In [17]:
# Exercise 5d: Test with multiple concurrent requests
# This simulates what happens when multiple users hit your server at once

import requests
import time
import concurrent.futures
import json

BASE = "http://localhost:8000"

prompts = [
    "What is Python?",
    "What is machine learning?",
    "What is an API?",
    "What is Docker?",
    "What is a neural network?"
]

def send_request(prompt):
    """Send a single request and return timing info."""
    start = time.time()
    r = requests.post(f"{BASE}/chat", json={
        "prompt": prompt,
        "max_tokens": 50
    })
    elapsed = time.time() - start
    result = r.json()
    return {
        "prompt": prompt[:30],
        "status": r.status_code,
        "tokens": result.get("tokens_generated", 0),
        "time": elapsed
    }

# Send requests concurrently
print("Sending 5 concurrent requests to the local server...")
print("=" * 60)

start_total = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(send_request, p) for p in prompts]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

total_time = time.time() - start_total

for r in sorted(results, key=lambda x: x["time"]):
    print(f"  {r['prompt']:<30} | {r['tokens']:>3} tokens | {r['time']:.2f}s | status={r['status']}")

print(f"\nAll 5 requests completed in {total_time:.2f}s total")
print("\nNote: Ollama processes requests sequentially by default,")
print("so concurrent requests are queued. In production, you would")
print("use vLLM or Triton for true concurrent processing.")

Sending 5 concurrent requests to the local server...
  What is Docker?                |   0 tokens | 0.00s | status=503
  What is an API?                |   0 tokens | 0.00s | status=503
  What is Python?                |   0 tokens | 0.02s | status=503
  What is machine learning?      |   0 tokens | 0.02s | status=503
  What is a neural network?      |   0 tokens | 0.02s | status=503

All 5 requests completed in 0.02s total

Note: Ollama processes requests sequentially by default,
so concurrent requests are queued. In production, you would
use vLLM or Triton for true concurrent processing.


In [18]:
# Exercise 5e: Clean up — stop the server
try:
    server_process.terminate()
    server_process.wait(timeout=5)
    print("Server stopped successfully.")
except Exception as e:
    print(f"Note: {e}")
    # Force kill if needed
    try:
        server_process.kill()
    except Exception:
        pass

Server stopped successfully.


---

### 🔧 Exercise 6: Model Management

In an on-premise deployment, you need to manage which models are available, monitor disk usage, and decide which models to keep loaded. Let us explore Ollama’s model management features.

In [19]:
# Exercise 6a: List and inspect available models
import requests
import json

def list_models():
    """List all models in Ollama with details."""
    r = requests.get("http://localhost:11434/api/tags")
    models = r.json().get("models", [])

    print(f"{'Model Name':<35} {'Size':>10} {'Quantisation':>15} {'Modified':>20}")
    print("=" * 85)

    total_size = 0
    for m in models:
        name = m["name"]
        size_gb = m.get("size", 0) / (1024**3)
        total_size += m.get("size", 0)
        # Get quantisation from details if available
        quant = m.get("details", {}).get("quantization_level", "unknown")
        modified = m.get("modified_at", "")[:10]

        print(f"  {name:<33} {size_gb:>8.2f} GB {quant:>15} {modified:>20}")

    print(f"\n  Total disk usage: {total_size / (1024**3):.2f} GB")
    return models

models = list_models()

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x12f521fa0>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [20]:
# Exercise 6b: Get detailed model information
import requests
import json

def show_model_info(model_name):
    """Get detailed information about a specific model."""
    r = requests.post("http://localhost:11434/api/show", json={"name": model_name})

    if r.status_code != 200:
        print(f"Model '{model_name}' not found.")
        return

    info = r.json()

    print(f"Model: {model_name}")
    print("=" * 60)

    # Model details
    details = info.get("details", {})
    print(f"  Family: {details.get('family', 'N/A')}")
    print(f"  Parameter Size: {details.get('parameter_size', 'N/A')}")
    print(f"  Quantisation: {details.get('quantization_level', 'N/A')}")
    print(f"  Format: {details.get('format', 'N/A')}")

    # Template (how the model expects prompts)
    template = info.get("template", "N/A")
    if len(template) > 200:
        template = template[:200] + "..."
    print(f"\n  Prompt Template: {template}")

# Show info for the model we downloaded
show_model_info("llama3.2:1b")

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/show (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x12f63cfb0>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [21]:
# Exercise 6c: Pull another model and compare
# Let's pull a slightly larger model to compare
# (Uncomment the pull command if you have bandwidth and want to try)

import requests
import json
import subprocess

print("Model Management Commands:")
print("=" * 60)
print()
print("To pull a new model:")
print("  ollama pull phi3:mini          # Microsoft Phi-3 Mini (~2.3 GB)")
print("  ollama pull gemma2:2b          # Google Gemma 2 2B (~1.6 GB)")
print("  ollama pull llama3.2:3b        # Meta Llama 3.2 3B (~2 GB)")
print()
print("To remove a model:")
print("  ollama rm <model-name>")
print()
print("To copy/rename a model:")
print("  ollama cp llama3.2:1b my-model")
print()

# Uncomment the next line to pull an additional model:
# subprocess.run(["ollama", "pull", "gemma2:2b"])

# Show current disk usage
print("Current model storage:")
print("-" * 40)
r = requests.get("http://localhost:11434/api/tags")
models = r.json().get("models", [])
for m in models:
    size_gb = m.get("size", 0) / (1024**3)
    print(f"  {m['name']:<30} {size_gb:.2f} GB")

total = sum(m.get("size", 0) for m in models) / (1024**3)
print(f"\n  Total: {total:.2f} GB")
print(f"\nTip: On-premise deployments need careful storage planning.")
print(f"A 70B model can use 40+ GB of disk space.")

Model Management Commands:

To pull a new model:
  ollama pull phi3:mini          # Microsoft Phi-3 Mini (~2.3 GB)
  ollama pull gemma2:2b          # Google Gemma 2 2B (~1.6 GB)
  ollama pull llama3.2:3b        # Meta Llama 3.2 3B (~2 GB)

To remove a model:
  ollama rm <model-name>

To copy/rename a model:
  ollama cp llama3.2:1b my-model

Current model storage:
----------------------------------------


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x12f6b0c50>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [22]:
# Exercise 6d: Memory monitoring during inference
# Understanding memory usage is critical for on-premise deployment

import psutil
import requests
import time
import json

def monitor_inference(prompt, model="llama3.2:1b"):
    """Run inference while monitoring system memory."""

    # Get baseline memory
    mem_before = psutil.virtual_memory()

    print(f"System Memory Before Inference:")
    print(f"  Total: {mem_before.total / (1024**3):.1f} GB")
    print(f"  Used:  {mem_before.used / (1024**3):.1f} GB ({mem_before.percent}%)")
    print(f"  Free:  {mem_before.available / (1024**3):.1f} GB")
    print()

    # Run inference
    print("Running inference...")
    start = time.time()
    r = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False
    }, timeout=120)
    elapsed = time.time() - start

    # Get memory after
    mem_after = psutil.virtual_memory()

    result = r.json()

    print(f"\nSystem Memory During/After Inference:")
    print(f"  Used:  {mem_after.used / (1024**3):.1f} GB ({mem_after.percent}%)")
    print(f"  Free:  {mem_after.available / (1024**3):.1f} GB")
    print(f"  Delta: {(mem_after.used - mem_before.used) / (1024**2):.0f} MB")
    print()
    print(f"Inference Results:")
    print(f"  Tokens generated: {result.get('eval_count', 'N/A')}")
    print(f"  Duration: {elapsed:.2f}s")

    return result

result = monitor_inference(
    "Write a short paragraph about the importance of monitoring system resources in production ML deployments."
)

print(f"\nResponse: {result['response'][:300]}...")

System Memory Before Inference:
  Total: 16.0 GB
  Used:  5.8 GB (80.8%)
  Free:  3.1 GB

Running inference...


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x12f73b830>: Failed to establish a new connection: [Errno 61] Connection refused'))

---

## Recap and Key Takeaways

### What We Covered

In this session, we explored **on-premise model deployment** — running ML models on your own hardware instead of in the cloud.

**Theory:**
1. **Why on-premise?** — Data privacy, latency, and cost at scale
2. **Linux + Nvidia GPUs** — The standard industry setup with frameworks like Triton, vLLM, and TorchServe
3. **Apple Silicon** — A viable alternative using unified memory, MLX framework, and Ollama
4. **Distributed inference** — Splitting large models across multiple devices
5. **Decision framework** — Choosing the right deployment target based on your needs

**Hands-on:**
1. **Ollama** — Installed and ran LLMs locally, used the REST API
2. **MLX** — Used Apple’s framework for inference on Apple Silicon (or learned about it)
3. **FastAPI server** — Built a local inference server with logging and rate limiting
4. **Model management** — Listed, inspected, and managed local models

### Key Takeaways

| Takeaway | Details |
|----------|--------|
| On-premise is about **control** | You control the data, the hardware, and the latency |
| **Ollama** makes local LLMs easy | One command to install, one command to run — great for development |
| **MLX** leverages Apple Silicon | Unified memory lets you run larger models on Mac than GPU VRAM would allow |
| **FastAPI + Ollama** = simple server | A production-ready pattern for on-premise deployment |
| Start simple, scale up | Ollama for dev → vLLM/Triton for production |

### What is Next?

In **Session 5**, we will cover **Retrieval-Augmented Generation (RAG)** — a technique for giving LLMs access to your own documents and data. RAG can run on **any** of the deployment targets we covered today:
- Cloud APIs
- Ollama on your laptop
- A production GPU server

The skills you learned today — running models locally, building API servers, managing models — are the foundation for everything that comes next.

💡 **Key Concept:** The best deployment is the one that meets your requirements. There is no one-size-fits-all answer. Understanding the trade-offs between cloud and on-premise deployment is what separates a data scientist from an ML engineer.

---

## Additional Resources

- [Ollama Documentation](https://github.com/ollama/ollama)
- [MLX Documentation](https://ml-explore.github.io/mlx/)
- [MLX Community Models on Hugging Face](https://huggingface.co/mlx-community)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [Nvidia Triton Inference Server](https://github.com/triton-inference-server/server)
- [vLLM Project](https://github.com/vllm-project/vllm)
- [TorchServe](https://pytorch.org/serve/)

---

*Session 4 of Module 5: Deep Learning in Production*  
*SMU Advanced Certificate in Generative AI and Deep Learning*